# LeetCode 1064 — Fixed Point bằng Python

**Link đề:** https://leetcode.com/problems/fixed-point/
**Độ khó:** Easy · **Chủ đề:** Array, Binary Search

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def fixedPoint(self, arr: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **điểm bất động** (fixed point): chỉ số `i` mà `arr[i] == i`.
2. Viết lời giải duyệt tuyến tính `O(n)` và thấy vì sao nó chưa tận dụng giả thiết "mảng đã sắp, phần tử phân biệt".
3. Khám phá insight: dãy `arr[i] - i` **không giảm** → dùng được **Binary Search** (tìm kiếm nhị phân).
4. Nắm khuôn binary search tìm **vị trí trái nhất** thỏa điều kiện (lower bound), trace `lo`, `hi`, `mid`.
5. Biết dùng `bisect` của thư viện chuẩn; chứng minh tính đúng; tự làm biến thể.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — tìm chỉ số nhỏ nhất mà giá trị bằng chỉ số

Cho mảng `arr` gồm các số nguyên **phân biệt**, đã **sắp tăng dần**. Trả về **chỉ số nhỏ nhất** `i` thỏa `arr[i] == i`. Nếu không có, trả về `-1`.

| `arr` | Output | Giải thích |
|---|:---:|---|
| `[-10,-5,0,3,7]` | `3` | `arr[3] = 3` |
| `[0,2,5,8,17]` | `0` | `arr[0] = 0` |
| `[-10,-5,3,4,7,9]` | `-1` | không có `i` nào thỏa |

**Constraints (theo hiểu biết):**
- `1 <= arr.length < 10^4`
- `-10^9 <= arr[i] <= 10^9`
- Các phần tử phân biệt, sắp tăng dần.

**Follow-up:** Cách `O(n)` rất dễ. Có cách nào tốt hơn không? → `O(log n)`.

In [1]:
arr = [-10, -5, 0, 3, 7]
for i, x in enumerate(arr):
    print(f"i={i}, arr[i]={x:>3}, bằng nhau? {x == i}")

i=0, arr[i]=-10, bằng nhau? False
i=1, arr[i]= -5, bằng nhau? False
i=2, arr[i]=  0, bằng nhau? False
i=3, arr[i]=  3, bằng nhau? True
i=4, arr[i]=  7, bằng nhau? False


## 2. Nền tảng Python và thuật toán cần biết

### 2.1. `enumerate` — vừa có chỉ số vừa có giá trị

`for i, x in enumerate(arr)` cho lần lượt `(0, arr[0]), (1, arr[1]), ...`. Gọn hơn `for i in range(len(arr)): x = arr[i]`.

### 2.2. Binary Search là gì?

Tìm kiếm trên **dữ liệu có thứ tự** bằng cách mỗi lần nhìn **phần tử giữa** và **loại bỏ một nửa** khoảng tìm kiếm.

Ví dụ đoán số trong `1..100`: hỏi "lớn hơn 50 không?" → còn 50 khả năng → hỏi tiếp… Sau khoảng `log2(100) ≈ 7` câu là tìm ra.

In [2]:
import math
for n in [10, 100, 10_000, 1_000_000_000]:
    print(f"n = {n:>13,}: duyệt tuyến tính ~{n:,} bước | binary search ~{math.ceil(math.log2(n))} bước")

n =            10: duyệt tuyến tính ~10 bước | binary search ~4 bước
n =           100: duyệt tuyến tính ~100 bước | binary search ~7 bước
n =        10,000: duyệt tuyến tính ~10,000 bước | binary search ~14 bước
n = 1,000,000,000: duyệt tuyến tính ~1,000,000,000 bước | binary search ~30 bước


### 2.3. Phép chia lấy nguyên `//` để tìm chỉ số giữa

`mid = (lo + hi) // 2` luôn là số nguyên, nằm trong `[lo, hi]`. Python không bị tràn số nên không cần mẹo `lo + (hi - lo) // 2` như C++/Java (vẫn có thể viết vậy cho quen).

In [3]:
for lo, hi in [(0, 4), (0, 5), (3, 3), (2, 7)]:
    mid = (lo + hi) // 2
    print(f"lo={lo}, hi={hi} -> mid={mid}, mid kiểu khác: {lo + (hi - lo) // 2}")

lo=0, hi=4 -> mid=2, mid kiểu khác: 2
lo=0, hi=5 -> mid=2, mid kiểu khác: 2
lo=3, hi=3 -> mid=3, mid kiểu khác: 3
lo=2, hi=7 -> mid=4, mid kiểu khác: 4


## 3. Cách 1 — Duyệt tuyến tính (baseline)

Đi từ trái sang phải; gặp `arr[i] == i` **đầu tiên** thì trả `i` ngay — đó chắc chắn là chỉ số nhỏ nhất. Hết mảng mà không gặp → `-1`.

In [4]:
def fixed_point_linear(arr: list[int]) -> int:
    for i, x in enumerate(arr):
        if x == i:
            return i          # gặp đầu tiên = nhỏ nhất
    return -1

print(fixed_point_linear([-10, -5, 0, 3, 7]))     # 3
print(fixed_point_linear([0, 2, 5, 8, 17]))       # 0
print(fixed_point_linear([-10, -5, 3, 4, 7, 9]))  # -1

3
0
-1


### Cải tiến nhỏ: dừng sớm khi `arr[i] > i`

Nếu `arr[i] > i` thì với mọi `j > i`, `arr[j] > j` cũng đúng (sẽ chứng minh ở mục 4). Vậy có thể `break` sớm. Nhưng worst case vẫn `O(n)` (ví dụ toàn số âm rất nhỏ).

In [5]:
def fixed_point_linear_early(arr: list[int]) -> int:
    for i, x in enumerate(arr):
        if x == i:
            return i
        if x > i:             # từ đây trở đi arr[j] > j
            return -1
    return -1

print(fixed_point_linear_early([-10, -5, 3, 4, 7, 9]))   # dừng ở i=2

-1


## 4. Nút thắt & insight: biến đổi để có dãy đơn điệu

Cách 1 bỏ phí giả thiết **"sắp tăng, phân biệt"**. Ta tìm tính chất đơn điệu để binary search.

**Quan sát:** các số nguyên phân biệt sắp tăng ⇒ `arr[i+1] >= arr[i] + 1`.

Đặt `diff[i] = arr[i] - i`. Khi đó:

```text
diff[i+1] = arr[i+1] - (i+1) >= arr[i] + 1 - i - 1 = diff[i]
```

⇒ dãy `diff` **không giảm**. Câu hỏi "tìm `i` nhỏ nhất với `arr[i] == i`" thành "tìm `i` nhỏ nhất với `diff[i] == 0`" trên một dãy không giảm — chính là bài **tìm vị trí trái nhất** của giá trị 0!

Ví dụ `[-10,-5,0,3,7]`: `diff = [-10,-6,-2,0,3]`.

In [6]:
for arr in [[-10, -5, 0, 3, 7], [0, 2, 5, 8, 17], [-10, -5, 3, 4, 7, 9]]:
    diff = [x - i for i, x in enumerate(arr)]
    print("arr =", arr, "\ndiff =", diff, "\n")

arr = [-10, -5, 0, 3, 7] 
diff = [-10, -6, -2, 0, 3] 

arr = [0, 2, 5, 8, 17] 
diff = [0, 1, 3, 5, 13] 

arr = [-10, -5, 3, 4, 7, 9] 
diff = [-10, -6, 1, 1, 3, 4] 



Hệ quả: các vị trí chia thành 2 vùng liền nhau:

```text
[  diff < 0 (arr[i] < i)  |  diff >= 0 (arr[i] >= i)  ]
```

Điểm bất động nhỏ nhất (nếu có) phải là **vị trí đầu tiên của vùng phải**. Ta binary search tìm ranh giới đó, rồi kiểm tra `arr[pos] == pos`.

**Chú ý:** nếu mảng có phần tử **trùng nhau** thì tính chất này **không còn** (ví dụ `[2,2,2]`: diff `[2,1,0]` giảm). Giả thiết "phân biệt" là then chốt.

## 5. Cách 2 (lời giải chính) — Binary Search tìm ranh giới trái nhất

Ý tưởng: giữ khoảng `[lo, hi]` còn nghi ngờ; `ans = -1`.

**Đoạn 1:** `lo, hi = 0, len(arr) - 1`.

**Đoạn 2:** `while lo <= hi:` lấy `mid = (lo + hi) // 2`.

**Đoạn 3 — quyết định:**
- `arr[mid] < mid` → `mid` và mọi chỉ số bên trái đều `diff < 0` → bỏ nửa trái: `lo = mid + 1`.
- `arr[mid] > mid` → `mid` không phải đáp án, và mọi chỉ số bên phải đều `diff > 0` → bỏ nửa phải: `hi = mid - 1`.
- `arr[mid] == mid` → `mid` là ứng viên; **ghi lại** `ans = mid` nhưng vẫn **tìm tiếp bên trái** để có chỉ số nhỏ hơn: `hi = mid - 1`.

**Đoạn 4:** trả `ans`.

In [7]:
def fixed_point_binary(arr: list[int]) -> int:
    lo, hi = 0, len(arr) - 1
    ans = -1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] < mid:
            lo = mid + 1          # đáp án (nếu có) nằm bên phải
        elif arr[mid] > mid:
            hi = mid - 1          # đáp án (nếu có) nằm bên trái
        else:
            ans = mid             # ứng viên; tìm tiếp bên trái
            hi = mid - 1
    return ans

print(fixed_point_binary([-10, -5, 0, 3, 7]))      # 3
print(fixed_point_binary([0, 2, 5, 8, 17]))        # 0
print(fixed_point_binary([-10, -5, 3, 4, 7, 9]))   # -1

3
0
-1


### Vì sao vẫn tìm tiếp bên trái khi đã gặp `arr[mid] == mid`?

Có thể có **nhiều** điểm bất động liên tiếp, ví dụ `[-1, 1, 2, 3, 4]`: mọi `i >= 1` đều thỏa. Lần nhìn đầu tiên `mid = 2` thỏa, nhưng đáp án nhỏ nhất là `1`.

(Thực ra khi `diff` đạt 0 thì các vị trí `diff == 0` tạo thành một đoạn liên tiếp — vì phân biệt + tăng.)

In [8]:
arr = [-1, 1, 2, 3, 4]
print("diff =", [x - i for i, x in enumerate(arr)])
print("binary:", fixed_point_binary(arr), "| linear:", fixed_point_linear(arr))

diff = [-1, 0, 0, 0, 0]
binary: 1 | linear: 1


## 6. Trace binary search

### Ví dụ `arr = [-10,-5,0,3,7]` (`diff = [-10,-6,-2,0,3]`)

| Vòng | `lo` | `hi` | `mid` | `arr[mid]` | So sánh | Hành động | `ans` |
|---:|:---:|:---:|:---:|:---:|---|---|:---:|
| 1 | 0 | 4 | 2 | 0 | `0 < 2` | `lo = 3` | -1 |
| 2 | 3 | 4 | 3 | 3 | `3 == 3` | `ans = 3`, `hi = 2` | 3 |
| — | 3 | 2 | | | `lo > hi` | dừng | **3** |

In [9]:
def trace_fixed_point(arr: list[int]) -> int:
    print("arr  =", arr)
    print("diff =", [x - i for i, x in enumerate(arr)])
    lo, hi, ans, step = 0, len(arr) - 1, -1, 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] < mid:
            act = f"arr[mid]={arr[mid]} < {mid} -> lo = {mid + 1}"
            lo = mid + 1
        elif arr[mid] > mid:
            act = f"arr[mid]={arr[mid]} > {mid} -> hi = {mid - 1}"
            hi = mid - 1
        else:
            ans = mid
            act = f"arr[mid]={arr[mid]} == {mid} -> ans = {mid}, hi = {mid - 1}"
            hi = mid - 1
        print(f"vòng {step}: mid={mid}: {act}")
        step += 1
    print(f"dừng vì lo={lo} > hi={hi}; ans = {ans}\n")
    return ans

trace_fixed_point([-10, -5, 0, 3, 7])
trace_fixed_point([-1, 1, 2, 3, 4])

arr  = [-10, -5, 0, 3, 7]
diff = [-10, -6, -2, 0, 3]
vòng 1: mid=2: arr[mid]=0 < 2 -> lo = 3
vòng 2: mid=3: arr[mid]=3 == 3 -> ans = 3, hi = 2
dừng vì lo=3 > hi=2; ans = 3

arr  = [-1, 1, 2, 3, 4]
diff = [-1, 0, 0, 0, 0]
vòng 1: mid=2: arr[mid]=2 == 2 -> ans = 2, hi = 1
vòng 2: mid=0: arr[mid]=-1 < 0 -> lo = 1
vòng 3: mid=1: arr[mid]=1 == 1 -> ans = 1, hi = 0
dừng vì lo=1 > hi=0; ans = 1



1

Trace ca **không có** đáp án, để thấy `ans` giữ nguyên `-1`:

In [10]:
trace_fixed_point([-10, -5, 3, 4, 7, 9])

arr  = [-10, -5, 3, 4, 7, 9]
diff = [-10, -6, 1, 1, 3, 4]
vòng 1: mid=2: arr[mid]=3 > 2 -> hi = 1
vòng 2: mid=0: arr[mid]=-10 < 0 -> lo = 1
vòng 3: mid=1: arr[mid]=-5 < 1 -> lo = 2
dừng vì lo=2 > hi=1; ans = -1



-1

## 7. Cách 3 — Binary search dạng "lower bound" (`lo < hi`)

Một khuôn khác rất phổ biến: tìm **vị trí đầu tiên** `pos` mà điều kiện `arr[pos] >= pos` đúng (hàm điều kiện dạng `F F F T T T`).

- `lo = 0`, `hi = n` (khoảng nửa mở `[lo, hi)`; `hi = n` nghĩa là "không có vị trí nào đúng").
- `while lo < hi`: nếu điều kiện đúng tại `mid` → `hi = mid` (giữ `mid` lại), ngược lại `lo = mid + 1`.
- Kết thúc `lo == hi == pos`. Kiểm tra `pos < n and arr[pos] == pos`.

In [11]:
def fixed_point_lower_bound(arr: list[int]) -> int:
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] >= mid:       # điều kiện "đã sang vùng phải"
            hi = mid
        else:
            lo = mid + 1
    return lo if lo < len(arr) and arr[lo] == lo else -1

print(fixed_point_lower_bound([-10, -5, 0, 3, 7]))
print(fixed_point_lower_bound([0, 2, 5, 8, 17]))
print(fixed_point_lower_bound([-10, -5, 3, 4, 7, 9]))
print(fixed_point_lower_bound([-3, -2, -1]))      # pos = 3 = n -> -1

3
0
-1
-1


## 8. Cách 4 — Dùng `bisect` trong thư viện chuẩn

`bisect.bisect_left(a, x)` trả về vị trí đầu tiên mà `a[pos] >= x` trên list đã sắp. Ta áp dụng cho dãy `diff` với `x = 0`.

Để **không tạo list `diff`** (tốn `O(n)`), Python 3.10+ có tham số `key=`: `bisect_left(arr_indices, 0, key=lambda i: arr[i] - i)`. Ở đây ta dùng `range(n)` làm "danh sách chỉ số" — `range` hỗ trợ truy cập theo chỉ số mà không tạo list.

In [12]:
from bisect import bisect_left

print(bisect_left([1, 3, 3, 5], 3))    # 1: vị trí đầu tiên có giá trị >= 3
print(bisect_left([1, 3, 3, 5], 4))    # 3
print(bisect_left([1, 3, 3, 5], 9))    # 4 = len -> không có

def fixed_point_bisect(arr: list[int]) -> int:
    n = len(arr)
    pos = bisect_left(range(n), 0, key=lambda i: arr[i] - i)
    return pos if pos < n and arr[pos] == pos else -1

print(fixed_point_bisect([-10, -5, 0, 3, 7]), fixed_point_bisect([-10, -5, 3, 4, 7, 9]))

1
3
4
3 -1


`lambda i: arr[i] - i` là **hàm vô danh** một dòng: nhận `i`, trả `arr[i] - i`. Tương đương:

```python
def key(i):
    return arr[i] - i
```

## 9. Code nộp LeetCode

### Bản binary search `lo <= hi` (lời giải chính)

In [13]:
class Solution:
    def fixedPoint(self, arr: list[int]) -> int:
        lo, hi = 0, len(arr) - 1
        ans = -1
        while lo <= hi:
            mid = (lo + hi) // 2
            if arr[mid] < mid:
                lo = mid + 1
            elif arr[mid] > mid:
                hi = mid - 1
            else:
                ans = mid
                hi = mid - 1
        return ans

sol = Solution()
assert sol.fixedPoint([-10, -5, 0, 3, 7]) == 3
assert sol.fixedPoint([0, 2, 5, 8, 17]) == 0
assert sol.fixedPoint([-10, -5, 3, 4, 7, 9]) == -1
print("Solution (binary search) PASS")

Solution (binary search) PASS


### Bản tuyến tính (đủ qua với `n < 10^4`)

In [14]:
class SolutionLinear:
    def fixedPoint(self, arr: list[int]) -> int:
        for i, x in enumerate(arr):
            if x == i:
                return i
        return -1

assert SolutionLinear().fixedPoint([-10, -5, 0, 3, 7]) == 3
assert SolutionLinear().fixedPoint([-10, -5, 3, 4, 7, 9]) == -1
print("SolutionLinear PASS")

SolutionLinear PASS


## 10. Vì sao binary search đúng?

**Bổ đề:** `diff[i] = arr[i] - i` không giảm (đã chứng minh ở mục 4 nhờ "nguyên, phân biệt, tăng").

**Invariant:** mọi điểm bất động có chỉ số **nhỏ hơn `ans`** (hoặc mọi điểm bất động, nếu `ans = -1`) đều nằm trong `[lo, hi]`.

- `arr[mid] < mid` (`diff[mid] < 0`): mọi `j <= mid` có `diff[j] <= diff[mid] < 0` → không phải điểm bất động → bỏ `[lo, mid]` an toàn.
- `arr[mid] > mid`: mọi `j >= mid` có `diff[j] > 0` → bỏ `[mid, hi]` an toàn.
- `arr[mid] == mid`: ghi `ans = mid`; các điểm bất động nhỏ hơn `mid` chỉ có thể ở `[lo, mid-1]` → `hi = mid - 1` giữ invariant.

**Dừng:** mỗi vòng khoảng `[lo, hi]` co lại ít nhất 1 → vòng lặp kết thúc. Khi `lo > hi`, khoảng rỗng ⇒ không còn điểm bất động nào nhỏ hơn `ans` ⇒ `ans` là nhỏ nhất (hoặc `-1` nếu chưa từng gặp).

## 11. Độ phức tạp

Gọi `n = len(arr)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Tuyến tính | `O(n)` | `O(1)` | đơn giản, không cần giả thiết sắp xếp |
| 2. Binary search `lo <= hi` | `O(log n)` | `O(1)` | tận dụng `diff` không giảm |
| 3. Lower bound `lo < hi` | `O(log n)` | `O(1)` | khuôn tổng quát `F...FT...T` |
| 4. `bisect` + `key` | `O(log n)` | `O(1)` | cần Python 3.10+ |

Đếm số lần đọc phần tử trên mảng lớn để thấy khác biệt:

In [15]:
def steps_linear(arr):
    steps = 0
    for i, x in enumerate(arr):
        steps += 1
        if x == i:
            return i, steps
    return -1, steps

def steps_binary(arr):
    lo, hi, ans, steps = 0, len(arr) - 1, -1, 0
    while lo <= hi:
        steps += 1
        mid = (lo + hi) // 2
        if arr[mid] < mid:
            lo = mid + 1
        elif arr[mid] > mid:
            hi = mid - 1
        else:
            ans, hi = mid, mid - 1
    return ans, steps

n = 9999
big = list(range(-n, 0)) [:n - 1] + [n - 1]   # n-1 số âm tăng dần, phần tử cuối = n-1 là điểm bất động
print("tuyến tính (kết quả, số bước):", steps_linear(big))
print("binary     (kết quả, số bước):", steps_binary(big))

tuyến tính (kết quả, số bước): (9998, 9999)
binary     (kết quả, số bước): (9998, 14)


Tuyến tính phải đi gần hết `9999` phần tử, binary search chỉ khoảng `log2(9999) ≈ 14` vòng.

## 12. Bộ test

Nhóm test:
1. Ba ví dụ đề.
2. Một phần tử: `[0]` → 0, `[5]` → -1, `[-1]` → -1.
3. Nhiều điểm bất động liên tiếp: `[-1,1,2,3,4]` → 1; `[0,1,2,3]` → 0.
4. Điểm bất động ở cuối: `[-5,-3,2]` → 2.
5. Toàn số âm / toàn số lớn.
6. Đối chiếu ngẫu nhiên 4 cách trên mảng phân biệt đã sắp.

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:26} PASS -> {actual}")

tests = [
    ("đề 1", [-10, -5, 0, 3, 7], 3),
    ("đề 2", [0, 2, 5, 8, 17], 0),
    ("đề 3", [-10, -5, 3, 4, 7, 9], -1),
    ("một phần tử [0]", [0], 0),
    ("một phần tử [5]", [5], -1),
    ("một phần tử [-1]", [-1], -1),
    ("đoạn bất động", [-1, 1, 2, 3, 4], 1),
    ("toàn bất động", [0, 1, 2, 3], 0),
    ("ở cuối", [-5, -3, 2], 2),
    ("toàn âm", [-9, -7, -3], -1),
    ("toàn lớn", [10, 20, 30], -1),
]
for name, arr, exp in tests:
    check(name, fixed_point_binary(arr), exp)
    assert fixed_point_linear(arr) == fixed_point_lower_bound(arr) == fixed_point_bisect(arr) == exp

đề 1                       PASS -> 3
đề 2                       PASS -> 0
đề 3                       PASS -> -1
một phần tử [0]            PASS -> 0
một phần tử [5]            PASS -> -1
một phần tử [-1]           PASS -> -1
đoạn bất động              PASS -> 1
toàn bất động              PASS -> 0
ở cuối                     PASS -> 2
toàn âm                    PASS -> -1
toàn lớn                   PASS -> -1


In [17]:
import random
random.seed(1064)
for _ in range(3000):
    n = random.randint(1, 30)
    arr = sorted(random.sample(range(-20, 40), n))     # sample -> phân biệt
    r = [fixed_point_linear(arr), fixed_point_binary(arr),
         fixed_point_lower_bound(arr), fixed_point_bisect(arr)]
    assert len(set(r)) == 1, (arr, r)
print("3000 mảng ngẫu nhiên: 4 cách khớp nhau")

3000 mảng ngẫu nhiên: 4 cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1: Gặp `arr[mid] == mid` là trả về ngay

```python
else:
    return mid
```

Sai khi có nhiều điểm bất động: `[-1,1,2,3,4]` trả `2` thay vì `1`.

In [18]:
def wrong_return_early(arr):
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] < mid:
            lo = mid + 1
        elif arr[mid] > mid:
            hi = mid - 1
        else:
            return mid
    return -1

print("sai :", wrong_return_early([-1, 1, 2, 3, 4]))
print("đúng:", fixed_point_binary([-1, 1, 2, 3, 4]))

sai : 2
đúng: 1


### Lỗi 2: Đi sai hướng — đảo `lo`/`hi`

Nếu `arr[mid] < mid` mà lại `hi = mid - 1`, ta vứt đúng nửa chứa đáp án. Mẹo nhớ: `arr[mid] < mid` nghĩa là "giá trị còn **tụt lại** so với chỉ số" → đáp án ở **bên phải**.

### Lỗi 3: Vòng lặp vô hạn khi trộn hai khuôn

Dùng `while lo <= hi` cùng `hi = mid` (thay vì `mid - 1`): khi `lo == hi == mid` và điều kiện đúng thì `hi` không đổi → lặp mãi. Quy tắc: `lo <= hi` đi với `mid ± 1`; `lo < hi` đi với `hi = mid`, `lo = mid + 1`.

### Lỗi 4: Áp dụng binary search khi có phần tử trùng

In [19]:
arr_dup = [-1, 0, 2, 2, 2, 2, 6]     # vi phạm giả thiết "phân biệt"
print("diff:", [x - i for i, x in enumerate(arr_dup)])     # không còn đơn điệu
print("linear:", fixed_point_linear(arr_dup), "| binary:", fixed_point_binary(arr_dup))

diff: [-1, -1, 0, -1, -2, -3, 0]
linear: 2 | binary: 6


Với phần tử trùng, `diff` có thể giảm, binary search có thể sai — ở ví dụ này nó trả `6` trong khi đáp án nhỏ nhất là `2` (`arr[2] = 2`). Luôn kiểm tra giả thiết trước khi dùng mẹo!

## 14. Mẹo nhận dạng pattern & biến thể

- Thấy "mảng **đã sắp**" + yêu cầu nhanh hơn `O(n)` → nghĩ tới **binary search**.
- Nếu điều kiện không trực tiếp đơn điệu, thử **biến đổi** (như `arr[i] - i`) để có dãy đơn điệu.
- "Chỉ số **nhỏ nhất** thỏa…" → binary search tìm **ranh giới trái** (ghi nhận rồi tìm tiếp bên trái, hoặc khuôn lower bound).

| Bài liên quan | Ý tưởng |
|---|---|
| 35. Search Insert Position | lower bound kinh điển |
| 34. Find First and Last Position | ranh giới trái + phải |
| 278. First Bad Version | `F...FT...T`, tìm `T` đầu tiên |
| 1539. Kth Missing Positive Number | biến đổi `arr[i] - (i + 1)` = số còn thiếu |

### Biến thể — Tìm **mọi** điểm bất động

Vì các vị trí `diff == 0` tạo đoạn liên tiếp, tìm ranh giới trái (`diff >= 0`) và ranh giới phải (`diff >= 1`) rồi lấy khoảng giữa.

In [20]:
def all_fixed_points(arr: list[int]) -> list[int]:
    n = len(arr)
    left = bisect_left(range(n), 0, key=lambda i: arr[i] - i)    # đầu tiên diff >= 0
    right = bisect_left(range(n), 1, key=lambda i: arr[i] - i)   # đầu tiên diff >= 1
    return list(range(left, right))

print(all_fixed_points([-1, 1, 2, 3, 7]))     # [1, 2, 3]
print(all_fixed_points([-10, -5, 3, 4, 7]))   # []

[1, 2, 3]
[]


## 15. Bài tập tự luyện (tăng dần)

**Bài 1 (tính tay):** `arr = [-3, -1, 1, 3, 5, 7]`. Viết dãy `diff`, rồi trace binary search (`lo`, `hi`, `mid`, `ans`).

**Bài 2 (điền chỗ trống):** hoàn thiện `ex_fixed_point` theo khung TODO.

**Bài 3 (viết lại):** `ex_first_ge(nums, target)` — vị trí đầu tiên `nums[i] >= target` trên mảng tăng (trả `len(nums)` nếu không có), **không** dùng `bisect`.

**Bài 4 (biến thể):** `ex_count_fixed_points(arr)` — đếm số điểm bất động trong `O(log n)`.

**Bài 5 (mini challenge — LeetCode 1539):** `ex_kth_missing(arr, k)` — `arr` là số nguyên dương tăng ngặt; trả về số nguyên dương thứ `k` **bị thiếu**. Gợi ý: số thiếu trước vị trí `i` là `arr[i] - (i + 1)`.

In [21]:
def ex_fixed_point(arr: list[int]) -> int:
    lo, hi, ans = 0, len(arr) - 1, -1
    while lo <= hi:
        mid = (lo + hi) // 2
        # TODO 1: arr[mid] < mid -> ?
        # TODO 2: arr[mid] > mid -> ?
        # TODO 3: bằng nhau -> ghi ans, tìm tiếp bên trái
        pass
    return ans

# assert ex_fixed_point([-10, -5, 0, 3, 7]) == 3
# assert ex_fixed_point([-1, 1, 2, 3, 4]) == 1

In [22]:
def ex_first_ge(nums: list[int], target: int) -> int:
    # TODO: khuôn lo, hi = 0, len(nums); while lo < hi: ...
    pass

# assert ex_first_ge([1, 3, 5, 7], 4) == 2
# assert ex_first_ge([1, 3, 5, 7], 8) == 4

In [23]:
def ex_count_fixed_points(arr: list[int]) -> int:
    # TODO: hai lần tìm ranh giới trên diff
    pass

# assert ex_count_fixed_points([-1, 1, 2, 3, 7]) == 3
# assert ex_count_fixed_points([5, 6]) == 0

In [24]:
def ex_kth_missing(arr: list[int], k: int) -> int:
    # TODO: tìm vị trí đầu tiên mà arr[i] - (i + 1) >= k, gọi là lo; đáp án = lo + k
    pass

# assert ex_kth_missing([2, 3, 4, 7, 11], 5) == 9
# assert ex_kth_missing([1, 2, 3, 4], 2) == 6

## 16. Gợi ý 3 mức

**Bài 1:** (1) `diff = arr[i] - i`. (2) `diff = [-3,-2,-1,0,1,2]`. (3) `mid=2 (1<2) → lo=3`; `mid=4 (5>4) → hi=3`; `mid=3 (3==3) → ans=3, hi=2`; dừng → 3.

**Bài 2:** (1) Giống mục 5. (2) `<` → `lo = mid + 1`; `>` → `hi = mid - 1`. (3) `else: ans = mid; hi = mid - 1`.

**Bài 3:** (1) Điều kiện `nums[mid] >= target` có dạng `F...FT...T`. (2) Đúng → `hi = mid`; sai → `lo = mid + 1`. (3) `return lo`.

**Bài 4:** (1) Đoạn `diff == 0` liên tiếp. (2) Ranh giới trái: đầu tiên `diff >= 0`; phải: đầu tiên `diff >= 1`. (3) Hiệu hai vị trí.

**Bài 5:** (1) `missing(i) = arr[i] - (i + 1)` không giảm. (2) Tìm `lo` đầu tiên có `missing(lo) >= k`. (3) Trước `lo` có `lo` số có mặt, nên đáp án `= lo + k`.

## 17. Đáp án tham khảo

Tự làm trước rồi mới xem. Bài 1: đáp án `3`.

In [25]:
def ans_fixed_point(arr):
    lo, hi, ans = 0, len(arr) - 1, -1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] < mid:
            lo = mid + 1
        elif arr[mid] > mid:
            hi = mid - 1
        else:
            ans = mid
            hi = mid - 1
    return ans

def ans_first_ge(nums, target):
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo

def ans_count_fixed_points(arr):
    def first_diff_ge(v):
        lo, hi = 0, len(arr)
        while lo < hi:
            mid = (lo + hi) // 2
            if arr[mid] - mid >= v:
                hi = mid
            else:
                lo = mid + 1
        return lo
    return first_diff_ge(1) - first_diff_ge(0)

def ans_kth_missing(arr, k):
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] - (mid + 1) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo + k

assert ans_fixed_point([-10, -5, 0, 3, 7]) == 3
assert ans_fixed_point([-1, 1, 2, 3, 4]) == 1
assert ans_fixed_point([-3, -1, 1, 3, 5, 7]) == 3
assert ans_first_ge([1, 3, 5, 7], 4) == 2 and ans_first_ge([1, 3, 5, 7], 8) == 4
assert ans_count_fixed_points([-1, 1, 2, 3, 7]) == 3 and ans_count_fixed_points([5, 6]) == 0
assert ans_kth_missing([2, 3, 4, 7, 11], 5) == 9 and ans_kth_missing([1, 2, 3, 4], 2) == 6
trace_fixed_point([-3, -1, 1, 3, 5, 7])
print("Đáp án tham khảo PASS toàn bộ test!")

arr  = [-3, -1, 1, 3, 5, 7]
diff = [-3, -2, -1, 0, 1, 2]
vòng 1: mid=2: arr[mid]=1 < 2 -> lo = 3
vòng 2: mid=4: arr[mid]=5 > 4 -> hi = 3
vòng 3: mid=3: arr[mid]=3 == 3 -> ans = 3, hi = 2
dừng vì lo=3 > hi=2; ans = 3

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def fixedPoint(self, arr: list[int]) -> int:
        lo, hi, ans = 0, len(arr) - 1, -1
        while lo <= hi:
            mid = (lo + hi) // 2
            if arr[mid] < mid:        # còn tụt lại -> sang phải
                lo = mid + 1
            elif arr[mid] > mid:      # đã vượt -> sang trái
                hi = mid - 1
            else:                     # ứng viên -> ghi lại, tìm tiếp bên trái
                ans, hi = mid, mid - 1
        return ans
```

**Checklist:**
- [ ] Hiểu vì sao `arr[i] - i` không giảm (cần **phân biệt + tăng + số nguyên**).
- [ ] Gặp `arr[mid] == mid` thì **không** trả ngay; tìm tiếp bên trái.
- [ ] Đúng hướng: `<` → phải, `>` → trái.
- [ ] Khuôn `lo <= hi` đi với `mid ± 1` (tránh lặp vô hạn).
- [ ] Trả `-1` khi không có.
- [ ] Time `O(log n)`, Space `O(1)`.